# NQ100 売買ルールバックテスト

Google Drive上のCSVを読み込み、東京時間の市場状態AIと売買シミュレーションを検証するためのColabノートブックです。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess
import sys
from pathlib import Path

# ColabでGitHubから開いた場合、必要に応じてリポジトリをcloneします。
REPO_DIR = Path('/content/NQ100')
if not REPO_DIR.exists():
    subprocess.run(
        ['git', 'clone', 'https://github.com/hon-daisuki/NQ100.git', str(REPO_DIR)],
        check=True,
    )

sys.path.insert(0, str(REPO_DIR))

In [ ]:
import pandas as pd
import numpy as np

from src.labels import add_future_return_label, add_direction_label
from src.backtest import build_long_short_returns, summarize_returns
from src.metrics import describe_time_range, missing_summary

## データ読み込み

In [ ]:
DATA_ROOT = Path('/content/drive')
CSV_CANDIDATES = [
    'USTEC_features_all.csv',
    'USTEC_1hour_features.csv',
]
PREFERRED_DIRS = [
    Path('/content/drive/MyDrive/CFD機械学習/backtest_ready'),
    Path('/content/drive/MyDrive/CFD機械学習'),
    Path('/content/drive/MyDrive'),
    DATA_ROOT,
]

def find_features_csv():
    checked = []
    for folder in PREFERRED_DIRS:
        for name in CSV_CANDIDATES:
            candidate = folder / name
            checked.append(str(candidate))
            if candidate.exists():
                return candidate

    matches = []
    for name in CSV_CANDIDATES:
        matches.extend(DATA_ROOT.rglob(name))
    if matches:
        return sorted(matches, key=lambda x: len(str(x)))[0]

    raise FileNotFoundError(
        '特徴量CSVが見つかりません。Driveに対象CSVがあるか、共有フォルダをMyDriveにショートカット追加してください。\n'
        + '探した候補:\n' + '\n'.join(checked)
    )

csv_path = find_features_csv()
print(f'Using CSV: {csv_path}')

df = pd.read_csv(csv_path)
df['日時'] = pd.to_datetime(df['日時'])
df = df.sort_values('日時').reset_index(drop=True)

describe_time_range(df)

In [ ]:
missing_summary(df)

## 東京時間フィルタとラベル作成

In [ ]:
df['hour'] = df['日時'].dt.hour
df_tokyo = df[(df['hour'] >= 8) & (df['hour'] <= 12)].copy()

df_tokyo = add_future_return_label(df_tokyo, horizon=4, threshold=0.005)
df_tokyo = add_direction_label(df_tokyo, return_col='future_return_4h', threshold=0.005)

df_tokyo[['日時', 'close', 'future_return_4h', 'is_trend', 'signal']].tail()

## 学習データ

In [ ]:
features_trend = [
    'return_1h', 'range', 'body', 'upper_wick', 'lower_wick',
    'hour', 'dayofweek', 'month', 'sma_5', 'sma_20',
    'macd', 'macd_signal', 'macd_hist', 'bb_width', 'rsi_14', 'atr_14'
]

df_model = df_tokyo.dropna(subset=features_trend + ['is_trend', 'future_return_4h']).copy()
split_index = int(len(df_model) * 0.8)

X_train = df_model[features_trend].iloc[:split_index]
X_test = df_model[features_trend].iloc[split_index:]
y_train = df_model['is_trend'].iloc[:split_index]
y_test = df_model['is_trend'].iloc[split_index:]

len(X_train), len(X_test), y_train.mean(), y_test.mean()

## LightGBM学習

In [ ]:
!pip -q install lightgbm

from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model_trend = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=31,
    class_weight='balanced',
    random_state=42,
)

model_trend.fit(X_train, y_train)
proba = model_trend.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

print('Accuracy =', accuracy_score(y_test, pred))
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred))

## Direction Model

Predict whether the future 4-hour move is down, flat, or up.

In [ ]:
direction_threshold = 0.005

df_model['direction_signal'] = 0
df_model.loc[df_model['future_return_4h'] > direction_threshold, 'direction_signal'] = 1
df_model.loc[df_model['future_return_4h'] < -direction_threshold, 'direction_signal'] = -1

# Convert -1/0/1 signals to 0/1/2 classes for LightGBM multiclass training.
signal_to_class = {-1: 0, 0: 1, 1: 2}
class_to_signal = {0: -1, 1: 0, 2: 1}

y_dir = df_model['direction_signal'].map(signal_to_class)
y_dir_train = y_dir.iloc[:split_index]
y_dir_test = y_dir.iloc[split_index:]

df_model['direction_signal'].value_counts(normalize=True).sort_index()


In [ ]:
model_dir = LGBMClassifier(
    objective='multiclass',
    num_class=3,
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=31,
    class_weight='balanced',
    random_state=42,
)

model_dir.fit(X_train, y_dir_train)
dir_proba = model_dir.predict_proba(X_test)
dir_class_pred = dir_proba.argmax(axis=1)
dir_signal_pred = pd.Series(dir_class_pred, index=X_test.index).map(class_to_signal)

print('Direction Accuracy =', accuracy_score(y_dir_test, dir_class_pred))
print(confusion_matrix(y_dir_test, dir_class_pred))
print(classification_report(
    y_dir_test,
    dir_class_pred,
    target_names=['down', 'flat', 'up'],
))


## Backtest With Predicted Direction

Use the trend model to select likely large-move periods, then use the direction model to choose long, short, or no trade. This cell does not use future direction as the trading signal.

In [ ]:
bt = df_model.iloc[split_index:].copy()
bt['trend_proba'] = proba
bt['dir_confidence'] = dir_proba.max(axis=1)
bt['pred_direction_signal'] = dir_signal_pred.values

trend_threshold = 0.6
direction_confidence_threshold = 0.45

bt['trade_signal'] = np.where(
    (bt['trend_proba'] >= trend_threshold)
    & (bt['dir_confidence'] >= direction_confidence_threshold),
    bt['pred_direction_signal'],
    0,
)

returns = build_long_short_returns(
    bt,
    signal_col='trade_signal',
    return_col='future_return_4h',
    cost_per_trade=0.0002,
)

print('Signal counts:')
print(bt['trade_signal'].value_counts().sort_index())
print('\nBacktest summary:')
summarize_returns(returns)


In [ ]:
datetime_col = df.columns[0]
bt[[datetime_col, 'close', 'future_return_4h', 'trend_proba', 'dir_confidence', 'pred_direction_signal', 'trade_signal']].tail(20)
